In [1]:
import sys

print("Python version:")
print(sys.version)

print("\nPython executable:")
print(sys.executable)

Python version:
3.11.15 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:12:53) [MSC v.1942 64 bit (AMD64)]

Python executable:
C:\Users\HOME\anaconda3\envs\healthcare-qml\python.exe


In [2]:
import pennylane as qml
import numpy as np
import pandas as pd
import sklearn
import matplotlib
import seaborn
import joblib

print("PennyLane:", qml.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Seaborn:", seaborn.__version__)

print("\nQML environment is ready.")

PennyLane: 0.45.1
NumPy: 2.4.6
Pandas: 3.0.5
Scikit-learn: 1.9.0
Matplotlib: 3.11.1
Seaborn: 0.13.2

QML environment is ready.


In [3]:
number_of_qubits = 2

quantum_device = qml.device(
    "default.qubit",
    wires=number_of_qubits
)

print("Quantum device created:")
print(quantum_device)

Quantum device created:
<default.qubit device (wires=2) at 0x198be10a490>


In [4]:
@qml.qnode(quantum_device)
def test_quantum_circuit(inputs):

    qml.RX(inputs[0], wires=0)
    qml.RY(inputs[1], wires=1)

    qml.CNOT(wires=[0, 1])

    return [
        qml.expval(qml.PauliZ(0)),
        qml.expval(qml.PauliZ(1))
    ]

In [5]:
test_inputs = np.array([0.5, 0.8])

circuit_output = test_quantum_circuit(
    test_inputs
)

print("Quantum circuit output:")
print(circuit_output)

Quantum circuit output:
[np.float64(0.8775825618903728), np.float64(0.6114176588750966)]


In [6]:
print(
    qml.draw(test_quantum_circuit)(
        test_inputs
    )
)

0: ──RX(0.50)─╭●─┤  <Z>
1: ──RY(0.80)─╰X─┤  <Z>


In [7]:
from pennylane import numpy as pnp

trainable_device = qml.device(
    "default.qubit",
    wires=2
)

@qml.qnode(
    trainable_device,
    interface="autograd"
)
def trainable_circuit(inputs, weights):

    qml.RY(inputs[0], wires=0)
    qml.RY(inputs[1], wires=1)

    qml.CNOT(wires=[0, 1])

    qml.RX(weights[0], wires=0)
    qml.RX(weights[1], wires=1)

    return qml.expval(
        qml.PauliZ(0)
    )

In [8]:
inputs = pnp.array(
    [0.4, 0.7],
    requires_grad=False
)

weights = pnp.array(
    [0.1, 0.2],
    requires_grad=True
)

output = trainable_circuit(
    inputs,
    weights
)

gradient_function = qml.grad(
    lambda parameters:
        trainable_circuit(
            inputs,
            parameters
        )
)

gradients = gradient_function(weights)

print("Circuit output:", output)
print("Parameter gradients:", gradients)

Circuit output: 0.9164595255079895
Parameter gradients: [-0.09195267  0.        ]


In [9]:
import pandas as pd

X = pd.read_excel(
    "clinical_feature_engineered_data.xlsx",
    sheet_name="Model_Features"
)

y = pd.read_excel(
    "clinical_feature_engineered_data.xlsx",
    sheet_name="Target"
).squeeze("columns")

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("Target classes:", y.nunique())

Feature shape: (500, 21)
Target shape: (500,)
Target classes: 12
